# Performance Rating Distribution & Calibration

## Objective
Analyze performance rating distribution across the organization to ensure fair, calibrated ratings.

## Key Questions
1. What is the overall performance rating distribution?
2. Are ratings consistent across departments and managers?
3. Which managers show rating inflation or deflation?
4. Is there demographic bias in performance ratings?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Load data
print("Loading performance data...")
workforce = pd.read_csv('../data/current_workforce.csv')
reviews = pd.read_csv('../data/performance_reviews.csv')
managers = pd.read_csv('../data/managers.csv')

print(f"Workforce: {len(workforce)} employees")
print(f"Performance reviews: {len(reviews)} records")
print(f"Managers: {len(managers)} managers")

## 1. Overall Performance Rating Distribution

In [ ]:
# Current rating distribution
rating_dist = workforce['latest_rating'].value_counts().sort_index()
rating_pct = (rating_dist / len(workforce) * 100).round(1)

print("\nCURRENT PERFORMANCE RATING DISTRIBUTION")
print("="*60)
print(f"{'Rating':<25} {'Count':<10} {'Percentage':<15}")
print("="*60)

rating_labels = {1: '1 - Does Not Meet', 2: '2 - Needs Improvement', 3: '3 - Meets Expectations',
                 4: '4 - Exceeds Expectations', 5: '5 - Outstanding'}

for rating in [5, 4, 3, 2, 1]:
    count = rating_dist.get(rating, 0)
    pct = rating_pct.get(rating, 0)
    print(f"{rating_labels[rating]:<25} {count:<10} {pct}%")

print("="*60)
print(f"\nAverage Rating: {workforce['latest_rating'].mean():.2f}")
print(f"Median Rating: {workforce['latest_rating'].median():.1f}")
print(f"Standard Deviation: {workforce['latest_rating'].std():.2f}")

In [ ]:
# Visualize rating distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = ['#e74c3c', '#e67e22', '#f39c12', '#3498db', '#2ecc71']
bars = ax1.bar(rating_dist.index, rating_dist.values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax1.set_xlabel('Performance Rating', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Employees', fontsize=12, fontweight='bold')
ax1.set_title('Performance Rating Distribution', fontsize=14, fontweight='bold')
ax1.set_xticks(range(1, 6))
ax1.set_xticklabels(['1\nDoes Not Meet', '2\nNeeds Improvement', '3\nMeets', '4\nExceeds', '5\nOutstanding'])
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}\n({height/len(workforce)*100:.1f}%)',
            ha='center', va='bottom', fontweight='bold')

# Comparison to target distribution (bell curve)
target_dist = pd.Series({1: 5, 2: 15, 3: 60, 4: 15, 5: 5})  # Example target %
actual_pct_series = pd.Series({i: (rating_dist.get(i, 0) / len(workforce) * 100) for i in range(1, 6)})

x = np.arange(1, 6)
width = 0.35
bars1 = ax2.bar(x - width/2, actual_pct_series, width, label='Actual', color='#3498db', alpha=0.8)
bars2 = ax2.bar(x + width/2, target_dist, width, label='Target (Example)', color='#95a5a6', alpha=0.8)

ax2.set_xlabel('Performance Rating', fontsize=12, fontweight='bold')
ax2.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
ax2.set_title('Actual vs. Target Distribution', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(['1', '2', '3', '4', '5'])
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Rating Distribution by Department

In [ ]:
# Department-level analysis
dept_ratings = workforce.groupby('department')['latest_rating'].agg(['mean', 'std', 'count']).round(2)
dept_ratings = dept_ratings.sort_values('mean', ascending=False)

print("\nPERFORMANCE RATINGS BY DEPARTMENT")
print("="*70)
print(f"{'Department':<20} {'Avg Rating':<15} {'Std Dev':<15} {'Employees':<10}")
print("="*70)
for dept, row in dept_ratings.iterrows():
    print(f"{dept:<20} {row['mean']:<15} {row['std']:<15} {int(row['count']):<10}")
print("="*70)

In [ ]:
# Visualize department ratings
fig, ax = plt.subplots(figsize=(14, 8))

# Box plot by department
dept_order = dept_ratings.index.tolist()
sns.boxplot(data=workforce, y='department', x='latest_rating', order=dept_order, palette='Set2', ax=ax)
ax.set_xlabel('Performance Rating', fontsize=12, fontweight='bold')
ax.set_ylabel('Department', fontsize=12, fontweight='bold')
ax.set_title('Performance Rating Distribution by Department', fontsize=14, fontweight='bold')
ax.axvline(workforce['latest_rating'].mean(), color='red', linestyle='--', linewidth=2, label=f'Company Avg ({workforce["latest_rating"].mean():.2f})')
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Manager Rating Patterns (Calibration Analysis)

In [ ]:
# Manager variance analysis
print("\nMANAGER RATING PATTERNS (Top 15 by Team Size)")
print("="*100)
print(f"{'Manager ID':<12} {'Department':<15} {'Team Size':<12} {'Avg Rating':<12} {'Std Dev':<12} {'Variance Flag':<15}")
print("="*100)

top_managers = managers.nlargest(15, 'team_size')

for _, mgr in top_managers.iterrows():
    # Flag managers with unusual patterns
    if mgr['avg_team_rating'] >= 4.0:
        flag = '⚠️ High Ratings'
    elif mgr['avg_team_rating'] <= 2.5:
        flag = '⚠️ Low Ratings'
    elif mgr['rating_std_dev'] > 1.2:
        flag = '⚠️ High Variance'
    elif mgr['rating_std_dev'] < 0.5:
        flag = '⚠️ Low Variance'
    else:
        flag = '✓ Normal'
    
    print(f"{mgr['manager_id']:<12} {mgr['department']:<15} {mgr['team_size']:<12} {mgr['avg_team_rating']:<12} {mgr['rating_std_dev']:<12} {flag:<15}")

print("="*100)
print("\n⚠️ High Ratings: May indicate rating inflation")
print("⚠️ Low Ratings: May indicate rating deflation or genuinely struggling team")
print("⚠️ High Variance: Inconsistent ratings across team")
print("⚠️ Low Variance: May indicate insufficient differentiation")

In [ ]:
# Visualize manager variance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: Avg rating vs std dev
colors = ['red' if x >= 4.0 else 'orange' if x <= 2.5 else 'green' for x in managers['avg_team_rating']]
ax1.scatter(managers['avg_team_rating'], managers['rating_std_dev'], 
           s=managers['team_size']*10, alpha=0.6, c=colors, edgecolors='black', linewidth=1)
ax1.set_xlabel('Average Team Rating', fontsize=12, fontweight='bold')
ax1.set_ylabel('Rating Standard Deviation', fontsize=12, fontweight='bold')
ax1.set_title('Manager Rating Patterns\n(Bubble size = team size)', fontsize=14, fontweight='bold')
ax1.axvline(workforce['latest_rating'].mean(), color='blue', linestyle='--', alpha=0.5, label='Company Avg')
ax1.axhline(0.8, color='gray', linestyle='--', alpha=0.5, label='Expected Std Dev')
ax1.legend()
ax1.grid(alpha=0.3)

# Distribution of manager avg ratings
ax2.hist(managers['avg_team_rating'], bins=15, color='#3498db', alpha=0.8, edgecolor='black')
ax2.axvline(workforce['latest_rating'].mean(), color='red', linestyle='--', linewidth=2, label='Company Avg')
ax2.set_xlabel('Manager Average Team Rating', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Managers', fontsize=12, fontweight='bold')
ax2.set_title('Distribution of Manager Average Ratings', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Rating Consistency Analysis (ICC - Intraclass Correlation)

In [ ]:
# Calculate ICC (simplified version)
# ICC measures consistency of ratings across managers

# Calculate between-manager variance and within-manager variance
overall_mean = workforce['latest_rating'].mean()
between_var = managers['avg_team_rating'].var()
within_var = workforce['latest_rating'].var()

# ICC(1) approximation
icc = between_var / (between_var + within_var)

print("\nRATING CONSISTENCY METRICS")
print("="*70)
print(f"Intraclass Correlation (ICC): {icc:.3f}")
print("\nInterpretation:")
print("  ICC < 0.40: Poor consistency (ratings vary widely by manager)")
print("  ICC 0.40-0.60: Fair consistency")
print("  ICC 0.60-0.80: Good consistency (TARGET RANGE)")
print("  ICC > 0.80: Excellent consistency (but may indicate lack of differentiation)")

if icc < 0.40:
    print("\n⚠️ ACTION NEEDED: Low consistency suggests calibration issues.")
    print("   → Conduct manager calibration sessions")
    print("   → Provide clearer rating guidelines")
elif icc > 0.80:
    print("\n⚠️ CONSIDERATION: Very high consistency may indicate insufficient differentiation.")
else:
    print("\n✓ Rating consistency is within acceptable range.")

print("="*70)

## 5. Demographic Analysis (Bias Check)

In [ ]:
# Rating by gender
gender_ratings = workforce.groupby('gender')['latest_rating'].agg(['mean', 'count']).round(2)

print("\nPERFORMANCE RATINGS BY GENDER")
print("="*60)
print(f"{'Gender':<15} {'Avg Rating':<15} {'Count':<10}")
print("="*60)
for gender, row in gender_ratings.iterrows():
    print(f"{gender:<15} {row['mean']:<15} {int(row['count']):<10}")
print("="*60)

# Rating by race
race_ratings = workforce.groupby('race')['latest_rating'].agg(['mean', 'count']).round(2)
race_ratings = race_ratings.sort_values('mean', ascending=False)

print("\nPERFORMANCE RATINGS BY RACE/ETHNICITY")
print("="*60)
print(f"{'Race/Ethnicity':<15} {'Avg Rating':<15} {'Count':<10}")
print("="*60)
for race, row in race_ratings.iterrows():
    print(f"{race:<15} {row['mean']:<15} {int(row['count']):<10}")
print("="*60)

print("\n📊 Bias Check: Review ratings by demographic groups for significant differences.")
print("   Differences > 0.3 points warrant further investigation.")

In [ ]:
# Visualize demographic analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gender
gender_ratings.plot(kind='bar', y='mean', ax=ax1, color='#3498db', alpha=0.8, legend=False)
ax1.set_xlabel('Gender', fontsize=12, fontweight='bold')
ax1.set_ylabel('Average Rating', fontsize=12, fontweight='bold')
ax1.set_title('Average Performance Rating by Gender', fontsize=14, fontweight='bold')
ax1.axhline(workforce['latest_rating'].mean(), color='red', linestyle='--', linewidth=2, label='Company Avg')
ax1.legend()
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)
ax1.grid(axis='y', alpha=0.3)

# Race
race_ratings.plot(kind='bar', y='mean', ax=ax2, color='#2ecc71', alpha=0.8, legend=False)
ax2.set_xlabel('Race/Ethnicity', fontsize=12, fontweight='bold')
ax2.set_ylabel('Average Rating', fontsize=12, fontweight='bold')
ax2.set_title('Average Performance Rating by Race/Ethnicity', fontsize=14, fontweight='bold')
ax2.axhline(workforce['latest_rating'].mean(), color='red', linestyle='--', linewidth=2, label='Company Avg')
ax2.legend()
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Performance Trends Over Time

In [ ]:
# Trend over review cycles
review_trend = reviews.groupby('review_cycle')['rating_numeric'].agg(['mean', 'count']).reset_index()
review_trend = review_trend.sort_values('review_cycle')

print("\nPERFORMANCE RATING TRENDS (By Review Cycle)")
print("="*60)
print(f"{'Review Cycle':<15} {'Avg Rating':<15} {'Reviews':<10}")
print("="*60)
for _, row in review_trend.iterrows():
    print(f"{row['review_cycle']:<15} {row['mean']:<15.2f} {int(row['count']):<10}")
print("="*60)

In [ ]:
# Visualize trend
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(review_trend['review_cycle'], review_trend['mean'], marker='o', linewidth=2, markersize=10, color='#3498db')
ax.set_xlabel('Review Cycle', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Rating', fontsize=12, fontweight='bold')
ax.set_title('Performance Rating Trend Over Time', fontsize=14, fontweight='bold')
ax.axhline(workforce['latest_rating'].mean(), color='red', linestyle='--', linewidth=2, label='Current Avg')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim([2.5, 4.0])

# Add value labels
for i, row in review_trend.iterrows():
    ax.text(row['review_cycle'], row['mean'] + 0.05, f"{row['mean']:.2f}", ha='center', fontweight='bold')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Key Recommendations

In [ ]:
print("\n" + "="*80)
print("KEY RECOMMENDATIONS")
print("="*80)

# Identify managers needing calibration
high_raters = managers[managers['avg_team_rating'] >= 4.0]
low_raters = managers[managers['avg_team_rating'] <= 2.5]

print("\n1. MANAGER CALIBRATION NEEDED")
if len(high_raters) > 0:
    print(f"   → {len(high_raters)} managers with high average ratings (≥4.0) - potential inflation")
    print(f"      Manager IDs: {', '.join(map(str, high_raters['manager_id'].tolist()[:5]))}...")
if len(low_raters) > 0:
    print(f"   → {len(low_raters)} managers with low average ratings (≤2.5) - investigate team issues or rating deflation")
    print(f"      Manager IDs: {', '.join(map(str, low_raters['manager_id'].tolist()[:5]))}...")

print("\n2. RATING DISTRIBUTION")
high_perf_pct = (workforce['latest_rating'] >= 4).sum() / len(workforce) * 100
low_perf_pct = (workforce['latest_rating'] <= 2).sum() / len(workforce) * 100
print(f"   → High performers (4-5): {high_perf_pct:.1f}% (Target: ~20%)")
print(f"   → Low performers (1-2): {low_perf_pct:.1f}% (Target: ~10-15%)")
if high_perf_pct > 30:
    print("   ⚠️ High performer % is elevated - may indicate rating inflation")

print("\n3. CALIBRATION SESSIONS")
print("   → Schedule bi-annual calibration meetings with department heads")
print("   → Review manager rating patterns before next review cycle")
print("   → Provide rating guidelines and examples to managers")

print("\n4. DEMOGRAPHIC EQUITY")
gender_diff = gender_ratings['mean'].max() - gender_ratings['mean'].min()
race_diff = race_ratings['mean'].max() - race_ratings['mean'].min()
print(f"   → Gender rating variance: {gender_diff:.2f} points")
print(f"   → Race rating variance: {race_diff:.2f} points")
if gender_diff > 0.3 or race_diff > 0.3:
    print("   ⚠️ Demographic differences warrant further investigation")
else:
    print("   ✓ No significant demographic bias detected")

print("\n" + "="*80)
print("Next Analysis: See '02_high_performer_retention.ipynb' for deep dive")
print("on high performer identification and retention strategies")
print("="*80)